# Trade Network Embeddings

We will generate embeddings of the oil trade network (HS 2709)

In [1]:
# Loading yearly trade data (HS code 2709) from 1988 - 2025

import numpy as np
import pandas as pd

edge_df = pd.read_csv("data/Oil Trade full graph.csv", encoding='latin-1', index_col=False)
edge_df

,source,target,weight,year
0,AUS,BRN,1.292020e+07,1988
1,AUS,CHN,4.353578e+06,1988
2,AUS,IDN,1.617891e+08,1988
3,AUS,IRN,6.861742e+06,1988
4,AUS,KWT,4.273570e+06,1988
...,...,...,...,...
43689,RWA,SWE,1.719343e+03,2025
43690,WSM,AUS,3.157200e+01,2025
43691,LCA,BRA,6.027934e+07,2025
43692,AIA,DOM,6.730000e+02,2025


In [2]:
start_year=1988
end_year=2026
dataframes = {}

for year in range(start_year,end_year):
    dataframes[year] = edge_df[edge_df["year"] == year].copy()
    dataframes[year] = dataframes[year].drop(columns="year")

In [3]:
nodes = set.union(set(edge_df['source']),set(edge_df['target']))
nodes_df = pd.DataFrame(set.union(set(edge_df['source']),set(edge_df['target'])), columns=["country"])
nodes_df = nodes_df.sort_values("country").reset_index(drop=True)

nodes_df["node_id"] = range(0, len(nodes_df))
id_to_idx = dict(zip(nodes_df["country"], nodes_df["node_id"]))
idx_to_id = dict(zip(nodes_df["node_id"].astype(str), nodes_df["country"]))
nodes_df

,country,node_id
0,ABW,0
1,AFG,1
2,AGO,2
3,AIA,3
4,ALB,4
...,...,...
240,YMD,240
241,YUG,241
242,ZAF,242
243,ZMB,243


In [210]:
# nodes_df.to_csv("/Users/kynesantos/node2vec/graph/countrylabels.csv", index=False)

In [4]:
dataframes_num = {}

for year in range(start_year,end_year):
    dataframes_num[year] = dataframes[year].copy()
    dataframes_num[year]["source"] = dataframes_num[year]["source"].map(id_to_idx)
    dataframes_num[year]["target"] = dataframes_num[year]["target"].map(id_to_idx)


In [220]:
# Saving and writing all of the edge lists

#for year in range(start_year,end_year):
    # dataframes_num[year].to_csv(
    # f"/Users/kynesantos/node2vec/graph/{year}_edgelist.txt", sep=" ", header=False,index=False)

In [5]:
def load_embeddings(path):
    """
    Loads a node2vec .emb file (word2vec text format):
    First line: num_nodes dim
    Each following line: node_id v1 v2 ... vd
    Returns dict {node_id: np.array}
    """
    embeddings = {}
    with open(path, 'r') as f:
        first_line = f.readline().split()
        num_nodes, dim = int(first_line[0]), int(first_line[1])
        for line in f:
            parts = line.split()
            node_id = int(parts[0])
            vec = np.array(parts[1:], dtype=float)
            embeddings[node_id] = vec
    return embeddings


In [6]:
# This function finds the k nearest neighbours to a country (node), given an embedding (path) and a metric

def k_nearest_neighbours(node, k, path, metric):
    embeddings = load_embeddings(path)
    countryid = id_to_idx[node]

    if countryid not in embeddings:
        raise ValueError(f"Node '{countryid}' not found in embeddings.")

    target_vec = embeddings[countryid]
    node_ids = list(embeddings.keys())

    # preallocate properly instead of distances[i] on an empty list
    distances = np.zeros(len(node_ids))
    for i, other_id in enumerate(node_ids):
        distances[i] = metric(target_vec, embeddings[other_id])

    # exclude the node itself before picking neighbours
    self_idx = node_ids.index(countryid)
    distances[self_idx] = -np.inf  # so it never gets picked as a "nearest" match by cos_sim

    nearest_idx = np.argpartition(-distances, k)[:k]  # top-k by similarity
    nearest_idx = nearest_idx[np.argsort(-distances[nearest_idx])]  # sort by similarity desc

    # return [(node_ids[i], distances[i]) for i in nearest_idx]
    return [idx_to_id[str(node_ids[i])]for i in nearest_idx]


In [223]:
k_nearest_neighbours('RUS', 10, '/Users/kynesantos/node2vec/emb/2013_trade.emb', cos_sim)

['BGR', 'BLR', 'ROU', 'SVK', 'POL', 'FIN', 'DEU', 'GRC', 'HUN', 'CZE']

In [164]:
k_nearest_neighbours('RUS', 10, '/Users/kynesantos/node2vec/emb/2014_trade.emb', cos_sim)

['SRB', 'CZE', 'HUN', 'SVK', 'LTU', 'HRV', 'IRL', 'DNK', 'BGR', 'UKR']

In [165]:
k_nearest_neighbours('RUS', 10, '/Users/kynesantos/node2vec/emb/2016_trade.emb', cos_sim)

['BLR', 'BGR', 'SRB', 'FIN', 'SVK', 'UKR', 'TUR', 'LTU', 'HUN', 'POL']

In [166]:
k_nearest_neighbours('RUS', 10, '/Users/kynesantos/node2vec/emb/2019_trade.emb', cos_sim)

['POL', 'GBR', 'HUN', 'SVK', 'PRT', 'BLR', 'URY', 'TUN', 'TUR', 'AGO']

In [199]:
k_nearest_neighbours('UKR', 10, '/Users/kynesantos/node2vec/emb/2013_trade.emb', cos_sim)

['CUB', 'UZB', 'LVA', 'PRK', 'GEO', 'LUX', 'ISL', 'BFA', 'ZMB', 'MDA']

In [200]:
print('2008: ' , k_nearest_neighbours('RUS', 10, '/Users/kynesantos/node2vec/emb/2008_trade.emb', cos_sim))
print('2010: ' , k_nearest_neighbours('RUS', 10, '/Users/kynesantos/node2vec/emb/2010_trade.emb', cos_sim))
print('2012: ' , k_nearest_neighbours('RUS', 10, '/Users/kynesantos/node2vec/emb/2012_trade.emb', cos_sim))
print('2014: ' , k_nearest_neighbours('RUS', 10, '/Users/kynesantos/node2vec/emb/2014_trade.emb', cos_sim))
print('2016: ' , k_nearest_neighbours('RUS', 10, '/Users/kynesantos/node2vec/emb/2016_trade.emb', cos_sim))
print('2018: ' , k_nearest_neighbours('RUS', 10, '/Users/kynesantos/node2vec/emb/2018_trade.emb', cos_sim))
print('2020: ' , k_nearest_neighbours('RUS', 10, '/Users/kynesantos/node2vec/emb/2020_trade.emb', cos_sim))
print('2022: ' , k_nearest_neighbours('RUS', 10, '/Users/kynesantos/node2vec/emb/2022_trade.emb', cos_sim))

2008:  ['POL', 'BLR', 'FIN', 'HUN', 'BGR', 'LTU', 'SVK', 'SWE', 'MKD', 'UKR']
2010:  ['LTU', 'SVK', 'UKR', 'SRB', 'TUR', 'HUN', 'BGR', 'FIN', 'MKD', 'DEU']
2012:  ['SWE', 'POL', 'HUN', 'BGR', 'BLR', 'LTU', 'SVK', 'FIN', 'DEU', 'CZE']
2014:  ['SRB', 'CZE', 'HUN', 'SVK', 'LTU', 'HRV', 'IRL', 'DNK', 'BGR', 'UKR']
2016:  ['BLR', 'BGR', 'SRB', 'FIN', 'SVK', 'UKR', 'TUR', 'LTU', 'HUN', 'POL']
2018:  ['LTU', 'BGR', 'SDN', 'IRL', 'POL', 'CUW', 'CYP', 'LBN', 'ARG', 'ISL']
2020:  ['HUN', 'FIN', 'BLR', 'POL', 'LTU', 'SVK', 'TCD', 'SRB', 'SSD', 'DEU']
2022:  ['BGR', 'TCD', 'SVK', 'GRC', 'SRB', 'CMR', 'LBY', 'DZA', 'POL', 'IND']


In [201]:
print('1988: ', k_nearest_neighbours('IRN', 10, '/Users/kynesantos/node2vec/emb/1988_trade.emb', cos_sim))
print('1990: ', k_nearest_neighbours('IRN', 10, '/Users/kynesantos/node2vec/emb/1990_trade.emb', cos_sim))
print('1992: ', k_nearest_neighbours('IRN', 10, '/Users/kynesantos/node2vec/emb/1992_trade.emb', cos_sim))
print('2008: ' , k_nearest_neighbours('IRN', 10, '/Users/kynesantos/node2vec/emb/2008_trade.emb', cos_sim))
print('2010: ' , k_nearest_neighbours('IRN', 10, '/Users/kynesantos/node2vec/emb/2010_trade.emb', cos_sim))
print('2012: ' , k_nearest_neighbours('IRN', 10, '/Users/kynesantos/node2vec/emb/2012_trade.emb', cos_sim))
print('2014: ' , k_nearest_neighbours('IRN', 10, '/Users/kynesantos/node2vec/emb/2014_trade.emb', cos_sim))
print('2016: ' , k_nearest_neighbours('IRN', 10, '/Users/kynesantos/node2vec/emb/2016_trade.emb', cos_sim))
print('2019: ' , k_nearest_neighbours('IRN', 10, '/Users/kynesantos/node2vec/emb/2019_trade.emb', cos_sim))

1988:  ['IND', 'MEX', 'QAT', 'VNM', 'GAB', 'IDN', 'EGY', 'CHN', 'JPN', 'ECU']
1990:  ['NZL', 'LKA', 'BGD', 'IDN', 'IND', 'VNM', 'AUS', 'QAT', 'CHN', 'KWT']
1992:  ['ROU', 'LKA', 'IRQ', 'BRA', 'ZWE', 'PAK', 'HKG', 'MDA', 'GNB', 'GEO']
2008:  ['LKA', 'MAR', 'OMN', 'PNG', 'MMR', 'ZMB', 'YEM', 'BGD', 'MNG', 'PHL']
2010:  ['EGY', 'LKA', 'ABW', 'ALB', 'AND', 'HKG', 'GNB', 'SYC', 'MAR', 'PRK']
2012:  ['LKA', 'SDN', 'JOR', 'IDN', 'ZMB', 'PNG', 'IRQ', 'TLS', 'KOR', 'MAR']
2014:  ['LKA', 'VNM', 'KWT', 'MYS', 'PAK', 'NZL', 'PNG', 'GIN', 'SAU', 'KOR']
2016:  ['COG', 'SSD', 'YEM', 'TCD', 'MNG', 'FJI', 'ZMB', 'BOL', 'BGD', 'ARG']
2019:  ['MYS', 'GAB', 'VEN', 'TCD', 'GHA', 'COG', 'LKA', 'SSD', 'PHL', 'JOR']


In [195]:
print('2008: ' , k_nearest_neighbours('UKR', 10, '/Users/kynesantos/node2vec/emb/2008_trade.emb', cos_sim))
print('2010: ' , k_nearest_neighbours('UKR', 10, '/Users/kynesantos/node2vec/emb/2010_trade.emb', cos_sim))
print('2012: ' , k_nearest_neighbours('UKR', 10, '/Users/kynesantos/node2vec/emb/2012_trade.emb', cos_sim))
print('2014: ' , k_nearest_neighbours('UKR', 10, '/Users/kynesantos/node2vec/emb/2014_trade.emb', cos_sim))
print('2016: ' , k_nearest_neighbours('UKR', 10, '/Users/kynesantos/node2vec/emb/2016_trade.emb', cos_sim))
print('2018: ' , k_nearest_neighbours('UKR', 10, '/Users/kynesantos/node2vec/emb/2018_trade.emb', cos_sim))
print('2020: ' , k_nearest_neighbours('UKR', 10, '/Users/kynesantos/node2vec/emb/2020_trade.emb', cos_sim))
print('2022: ' , k_nearest_neighbours('UKR', 10, '/Users/kynesantos/node2vec/emb/2022_trade.emb', cos_sim))

2008:  ['CZE', 'SVK', 'HUN', 'LTU', 'SRB', 'BGR', 'FIN', 'MKD', 'ROU', 'SWE']
2010:  ['SVK', 'LTU', 'TUR', 'CZE', 'BGR', 'HRV', 'HUN', 'FIN', 'POL', 'BLR']
2012:  ['HRV', 'AUT', 'CHE', 'BIH', 'MNE', 'TUR', 'MKD', 'LVA', 'TUN', 'ALB']
2014:  ['GEO', 'MLT', 'BIH', 'GUM', 'MNE', 'SYR', 'BMU', 'ISL', 'CPV', 'GIB']
2016:  ['TUN', 'SRB', 'TUR', 'EST', 'BRB', 'BIH', 'BEN', 'MAR', 'TJK', 'LVA']
2018:  ['TUN', 'SRB', 'CUW', 'CYP', 'FRA', 'LVA', 'PSE', 'HRV', 'DOM', 'BIH']
2020:  ['EST', 'MDA', 'TUN', 'TKM', 'PSE', 'SUR', 'CYP', 'FRO', 'SVN', 'BRB']
2022:  ['ALB', 'CHE', 'UZB', 'PRK', 'VEN', 'MLT', 'MAR', 'TKM', 'SEN', 'TGO']


In [185]:
print('1988: ' , k_nearest_neighbours('USA', 10, '/Users/kynesantos/node2vec/emb/1988_trade.emb', cos_sim))
print('1990: ' , k_nearest_neighbours('USA', 10, '/Users/kynesantos/node2vec/emb/1990_trade.emb', cos_sim))
print('1992: ' , k_nearest_neighbours('USA', 10, '/Users/kynesantos/node2vec/emb/1992_trade.emb', cos_sim))
print('2008: ' , k_nearest_neighbours('USA', 10, '/Users/kynesantos/node2vec/emb/2008_trade.emb', cos_sim))
print('2010: ' , k_nearest_neighbours('USA', 10, '/Users/kynesantos/node2vec/emb/2010_trade.emb', cos_sim))
print('2012: ' , k_nearest_neighbours('USA', 10, '/Users/kynesantos/node2vec/emb/2012_trade.emb', cos_sim))
print('2014: ' , k_nearest_neighbours('USA', 10, '/Users/kynesantos/node2vec/emb/2014_trade.emb', cos_sim))
print('2016: ' , k_nearest_neighbours('USA', 10, '/Users/kynesantos/node2vec/emb/2016_trade.emb', cos_sim))
print('2019: ' , k_nearest_neighbours('USA', 10, '/Users/kynesantos/node2vec/emb/2019_trade.emb', cos_sim))

1988:  ['AUS', 'SGP', 'MYS', 'BRN', 'OMN', 'KWT', 'PHL', 'THA', 'KOR', 'ECU']
1990:  ['MEX', 'FRA', 'NLD', 'ISR', 'JAM', 'DOM', 'ITA', 'PHL', 'ESP', 'HND']
1992:  ['AGO', 'VEN', 'GAB', 'COL', 'TTO', 'COG', 'NGA', 'SAU', 'ARG', 'PER']
2008:  ['JAM', 'TCD', 'GNQ', 'CMR', 'DZA', 'IRQ', 'TJK', 'PRT', 'COG', 'GAB']
2010:  ['TCD', 'VEN', 'IRQ', 'URY', 'NIC', 'ZMB', 'SHN', 'MAR', 'JAM', 'MOZ']
2012:  ['MAR', 'LKA', 'JOR', 'IND', 'IRN', 'ZAF', 'MRT', 'CHN', 'KOR', 'SEN']
2014:  ['VEN', 'IRQ', 'TCD', 'JAM', 'IND', 'MAR', 'PAK', 'JOR', 'KWT', 'VNM']
2016:  ['CAN', 'VEN', 'CUB', 'SAU', 'CUW', 'TCD', 'KWT', 'JOR', 'IDN', 'ZMB']
2019:  ['VEN', 'CIV', 'URY', 'IRL', 'NGA', 'IRQ', 'CAN', 'NIC', 'GBR', 'GHA']


In [186]:
print('1988: ' , k_nearest_neighbours('IRQ', 10, '/Users/kynesantos/node2vec/emb/1988_trade.emb', cos_sim))
print('1990: ' , k_nearest_neighbours('IRQ', 10, '/Users/kynesantos/node2vec/emb/1990_trade.emb', cos_sim))
print('1992: ' , k_nearest_neighbours('IRQ', 10, '/Users/kynesantos/node2vec/emb/1992_trade.emb', cos_sim))
print('2008: ' , k_nearest_neighbours('IRQ', 10, '/Users/kynesantos/node2vec/emb/2008_trade.emb', cos_sim))
print('2010: ' , k_nearest_neighbours('IRQ', 10, '/Users/kynesantos/node2vec/emb/2010_trade.emb', cos_sim))
print('2012: ' , k_nearest_neighbours('IRQ', 10, '/Users/kynesantos/node2vec/emb/2012_trade.emb', cos_sim))
print('2014: ' , k_nearest_neighbours('IRQ', 10, '/Users/kynesantos/node2vec/emb/2014_trade.emb', cos_sim))
print('2016: ' , k_nearest_neighbours('IRQ', 10, '/Users/kynesantos/node2vec/emb/2016_trade.emb', cos_sim))
print('2019: ' , k_nearest_neighbours('IRQ', 10, '/Users/kynesantos/node2vec/emb/2019_trade.emb', cos_sim))

1988:  ['CAN', 'CMR', 'COG', 'BGR', 'NLD', 'GIN', 'LBN', 'BEL', 'HKG', 'DNK']
1990:  ['EGY', 'BRA', 'TUR', 'SAU', 'ROU', 'IND', 'VNM', 'PAK', 'GRC', 'LKA']
1992:  ['MDA', 'UKR', 'PAK', 'GEO', 'HKG', 'GNB', 'CSK', 'ABW', 'MDG', 'LBR']
2008:  ['CIV', 'NGA', 'GHA', 'GAB', 'SEN', 'IND', 'CMR', 'PRT', 'CAN', 'SLE']
2010:  ['ZMB', 'VNM', 'KWT', 'MDV', 'MOZ', 'JOR', 'LKA', 'BLZ', 'ZAF', 'BOL']
2012:  ['AGO', 'SDN', 'ZAF', 'LKA', 'IRN', 'PRK', 'AFG', 'EGY', 'JOR', 'LAO']
2014:  ['TCD', 'JOR', 'MAR', 'PAK', 'SAU', 'KWT', 'IND', 'VEN', 'COD', 'JAM']
2016:  ['EGY', 'GNQ', 'CMR', 'BGD', 'WSM', 'CUW', 'GRC', 'MNG', 'ZMB', 'MDV']
2019:  ['VEN', 'TCD', 'GHA', 'GRC', 'IRN', 'MEX', 'GAB', 'MYS', 'LKA', 'CUB']


In [188]:
print('1988: ' , k_nearest_neighbours('CAN', 10, '/Users/kynesantos/node2vec/emb/1988_trade.emb', cos_sim))
print('1990: ' , k_nearest_neighbours('CAN', 10, '/Users/kynesantos/node2vec/emb/1990_trade.emb', cos_sim))
print('1992: ' , k_nearest_neighbours('CAN', 10, '/Users/kynesantos/node2vec/emb/1992_trade.emb', cos_sim))
print('2008: ' , k_nearest_neighbours('CAN', 10, '/Users/kynesantos/node2vec/emb/2008_trade.emb', cos_sim))
print('2010: ' , k_nearest_neighbours('CAN', 10, '/Users/kynesantos/node2vec/emb/2010_trade.emb', cos_sim))
print('2012: ' , k_nearest_neighbours('CAN', 10, '/Users/kynesantos/node2vec/emb/2012_trade.emb', cos_sim))
print('2014: ' , k_nearest_neighbours('CAN', 10, '/Users/kynesantos/node2vec/emb/2014_trade.emb', cos_sim))
print('2016: ' , k_nearest_neighbours('CAN', 10, '/Users/kynesantos/node2vec/emb/2016_trade.emb', cos_sim))
print('2019: ' , k_nearest_neighbours('CAN', 10, '/Users/kynesantos/node2vec/emb/2019_trade.emb', cos_sim))

1988:  ['HKG', 'CMR', 'FRA', 'BEL', 'IRQ', 'COG', 'BGR', 'GIN', 'LBN', 'NLD']
1990:  ['NGA', 'GBR', 'NOR', 'CHE', 'SUN', 'YEM', 'VEN', 'DEU', 'AGO', 'SYR']
1992:  ['GBR', 'FIN', 'IRL', 'NOR', 'DNK', 'LVA', 'SWE', 'BGR', 'CMR', 'MRT']
2008:  ['TCD', 'DZA', 'JAM', 'GAB', 'SEN', 'CIV', 'GHA', 'CMR', 'DOM', 'AIA']
2010:  ['PRT', 'FRA', 'CIV', 'GIN', 'IRL', 'GEO', 'TUN', 'DZA', 'UZB', 'SVN']
2012:  ['TCD', 'DZA', 'GHA', 'SEN', 'SPM', 'IRL', 'NPL', 'MDA', 'MAR', 'CYP']
2014:  ['NIC', 'DOM', 'TCD', 'VEN', 'JAM', 'MAR', 'ZAF', 'EGY', 'URY', 'CUW']
2016:  ['VEN', 'USA', 'CUW', 'ZAF', 'CUB', 'SAU', 'TCD', 'KWT', 'IDN', 'SEN']
2019:  ['DOM', 'VEN', 'MLT', 'FJI', 'JOR', 'NIC', 'IRQ', 'ZMB', 'MWI', 'TCD']


In [170]:
print(k_nearest_neighbours('CHN', 10, '/Users/kynesantos/node2vec/emb/2013_trade.emb', cos_sim))
print(k_nearest_neighbours('CHN', 10, '/Users/kynesantos/node2vec/emb/2014_trade.emb', cos_sim))
print(k_nearest_neighbours('CHN', 10, '/Users/kynesantos/node2vec/emb/2016_trade.emb', cos_sim))
print(k_nearest_neighbours('CHN', 10, '/Users/kynesantos/node2vec/emb/2019_trade.emb', cos_sim))

['SSD', 'IRN', 'YEM', 'LKA', 'TLS', 'COG', 'OMN', 'IND', 'PNG', 'IRQ']
['YEM', 'COG', 'SSD', 'SDN', 'ZAF', 'COD', 'AGO', 'OMN', 'PNG', 'MAR']
['COG', 'OMN', 'GHA', 'SSD', 'AGO', 'IRN', 'ARG', 'HKG', 'URY', 'MNG']
['OMN', 'COG', 'SSD', 'AGO', 'IND', 'BRA', 'URY', 'PRT', 'VEN', 'IRN']


In [106]:
def cos_sim(a,b):
    # computes the cosine similarity between 2 vectors
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def euc_dist(a,b):
    # computes the euclidean distance between 2 vectors
    return np.linalg.norm(a - b)